# DELCODE Resting-State Scan Date Extraction (ZIP-safe)

This notebook extracts scan dates directly from DICOM files stored inside `.zip` patient archives, without modifying original files.

It targets visit folders `M0` to `M60` and resting-state series names containing `RestingState` (with preference for exact `4-dzne_RestingState_3_5iso`), then exports an Excel workbook with separate sheets for `T1_01` and `T1_02`.

## Requirements
- Conda environment: `ad-early-detection`
- Python packages: `pydicom`, `pandas`, `openpyxl`

## Output
- One Excel workbook containing:
  - Sheet `T1_01`
  - Sheet `T1_02`
- Columns include pseudonym, visit, run, scan date (`YYYY-MM-DD`), source archive, and QC flags.

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable
import zipfile

import concurrent.futures as cf
from concurrent.futures import ProcessPoolExecutor

import pandas as pd
import pydicom
from pydicom.dataset import Dataset
from tqdm.auto import tqdm

# ===== User-configurable paths =====
ALL_DATA_ROOT = Path("/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Projects/Delcode/all_data")
OUTPUT_XLSX = Path("/dss/dsshome1/0A/di54lup/DELCODE/DCM-SCAN-DATES/restingstate_scan_dates_M0_M60.xlsx")

# Visits to include and sort order
VISIT_ORDER = ["M0", "M12", "M24", "M36", "M48", "M60"]
VISIT_RANK = {visit: idx for idx, visit in enumerate(VISIT_ORDER)}

# Sequence matching preferences
TARGET_EXACT = "4-dzne_RestingState_3_5iso"
TARGET_CONTAINS = "restingstate"

# Only these runs go to final workbook sheets
RUNS_OF_INTEREST = {"T1_01", "T1_02"}

DATE_TAGS_PRIORITY = [
    "AcquisitionDateTime",  # (0008,002A)
    "AcquisitionDate",      # (0008,0022)
    "SeriesDate",           # (0008,0021)
    "StudyDate",            # (0008,0020)
    "ContentDate",          # (0008,0023)
    "InstanceCreationDate", # (0008,0012)
]

# Header-only read: no pixel data
DICOM_SPECIFIC_TAGS = [
    "SeriesDescription",
    "ProtocolName",
    "SequenceName",
    "SeriesInstanceUID",
    "AcquisitionDateTime",
    "AcquisitionDate",
    "SeriesDate",
    "StudyDate",
    "ContentDate",
    "InstanceCreationDate",
]

print(f"All data root: {ALL_DATA_ROOT}")
print(f"Output file  : {OUTPUT_XLSX}")
print("Ready.")

All data root: /dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Projects/Delcode/all_data
Output file  : /dss/dsshome1/0A/di54lup/DELCODE/Delcode-Scan-Dates/restingstate_scan_dates_M0_M60.xlsx
Ready.


/dss/dsshome1/0A/di54lup/miniconda3/envs/ad-early-detection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ZIP_NAME_RE = re.compile(r"^(?P<pseudonym>[0-9a-fA-F]+)[-_](?P<visit>M\d+)[-_](?P<run>T\d+_\d+)\.zip$")

@dataclass
class ArchiveRecord:
    pseudonym: str
    visit: str
    run: str
    zip_path: Path


def iter_visit_dirs(root: Path) -> Iterable[Path]:
    for path in sorted(root.iterdir()):
        if not path.is_dir():
            continue
        if any(v in path.name for v in VISIT_ORDER):
            yield path


def parse_zip_name(zip_path: Path) -> ArchiveRecord | None:
    m = ZIP_NAME_RE.match(zip_path.name)
    if not m:
        return None
    visit = m.group("visit")
    run = m.group("run")
    if visit not in VISIT_RANK:
        return None
    if run not in RUNS_OF_INTEREST:
        return None
    return ArchiveRecord(
        pseudonym=m.group("pseudonym"),
        visit=visit,
        run=run,
        zip_path=zip_path,
    )


def normalize_text(value: str | None) -> str:
    return (value or "").strip().lower().replace(" ", "")


_SKIP_EXTS = frozenset((".txt", ".json", ".xml", ".csv", ".log"))

def is_target_series(ds: Dataset) -> tuple[bool, str]:
    candidates = [
        str(getattr(ds, "SeriesDescription", "") or ""),
        str(getattr(ds, "ProtocolName", "") or ""),
        str(getattr(ds, "SequenceName", "") or ""),
    ]
    raw = " | ".join(candidates)
    normalized = normalize_text(raw)

    exact_norm = normalize_text(TARGET_EXACT)
    if exact_norm and exact_norm in normalized:
        return True, "exact_or_strong_match"

    if TARGET_CONTAINS in normalized:
        return True, "contains_restingstate"

    return False, ""


def parse_dicom_date(ds: Dataset) -> tuple[str | None, str | None]:
    for tag in DATE_TAGS_PRIORITY:
        value = getattr(ds, tag, None)
        if value is None:
            continue
        txt = str(value).strip()
        if not txt:
            continue
        date_part = txt[:8]
        if len(date_part) == 8 and date_part.isdigit():
            try:
                parsed = datetime.strptime(date_part, "%Y%m%d").date().isoformat()
                return parsed, tag
            except ValueError:
                pass
    return None, None


def _make_row(rec: ArchiveRecord, ds: Dataset | None, member: str | None,
              match_type: str | None, qc_flag: str = "") -> dict:
    """Build one output row — avoids repeating the dict literal everywhere."""
    if ds is not None:
        scan_date, chosen_tag = parse_dicom_date(ds)
        return {
            "pseudonym": rec.pseudonym, "visit": rec.visit, "run": rec.run,
            "scan_date": scan_date,
            "date_source_tag": chosen_tag,
            "match_type": match_type,
            "series_description": str(getattr(ds, "SeriesDescription", "") or ""),
            "protocol_name": str(getattr(ds, "ProtocolName", "") or ""),
            "sequence_name": str(getattr(ds, "SequenceName", "") or ""),
            "series_instance_uid": str(getattr(ds, "SeriesInstanceUID", "") or ""),
            "source_zip": str(rec.zip_path),
            "source_member": member,
            "qc_flag": "missing_date" if scan_date is None else qc_flag,
        }
    return {
        "pseudonym": rec.pseudonym, "visit": rec.visit, "run": rec.run,
        "scan_date": None, "date_source_tag": None, "match_type": None,
        "series_description": None, "protocol_name": None, "sequence_name": None,
        "series_instance_uid": None,
        "source_zip": str(rec.zip_path), "source_member": None,
        "qc_flag": qc_flag,
    }


def extract_record_from_zip(rec: ArchiveRecord) -> list[dict]:
    """
    Optimized extraction: 2-stage approach.
    Stage 1: Filter ZIP member paths for 'rest' keyword — if found, read only
             the FIRST matching DICOM (benchmarked: 100% hit rate, ~15x faster).
    Stage 2: Fallback full scan with capped reads if path filter misses.
    """
    try:
        with zipfile.ZipFile(rec.zip_path, "r") as zf:
            members = [m for m in zf.namelist() if not m.endswith("/")]

            # --- Stage 1: path-based pre-filter (near-instant, ~180 matches/zip) ---
            rest_members = [
                m for m in members
                if "rest" in m.lower() and not any(m.lower().endswith(e) for e in _SKIP_EXTS)
            ]

            if rest_members:
                # Read 1 DICOM from each unique series in the rest-filtered set
                seen_series: set[str] = set()
                rows: list[dict] = []
                for member in rest_members:
                    try:
                        with zf.open(member) as f:
                            ds = pydicom.dcmread(
                                f, stop_before_pixels=True, force=True,
                                specific_tags=DICOM_SPECIFIC_TAGS,
                            )
                    except Exception:
                        continue

                    is_target, match_type = is_target_series(ds)
                    if not is_target:
                        continue

                    series_uid = str(getattr(ds, "SeriesInstanceUID", "") or "")
                    if series_uid and series_uid in seen_series:
                        continue
                    if series_uid:
                        seen_series.add(series_uid)

                    rows.append(_make_row(rec, ds, member, match_type))
                    break  # Early exit: we only need 1 valid resting-state date

                if rows:
                    return rows

            # --- Stage 2: fallback full scan (capped at 30 reads) ---
            MAX_FALLBACK_READS = 30
            fallback_reads = 0
            for member in members:
                lower = member.lower()
                if any(lower.endswith(e) for e in _SKIP_EXTS) or lower.endswith("/"):
                    continue
                try:
                    with zf.open(member) as f:
                        ds = pydicom.dcmread(
                            f, stop_before_pixels=True, force=True,
                            specific_tags=DICOM_SPECIFIC_TAGS,
                        )
                except Exception:
                    continue

                fallback_reads += 1
                is_target, match_type = is_target_series(ds)
                if is_target:
                    return [_make_row(rec, ds, member, match_type, qc_flag="fallback_scan")]

                if fallback_reads >= MAX_FALLBACK_READS:
                    break

            return [_make_row(rec, None, None, None, qc_flag="restingstate_not_found")]

    except Exception as exc:
        return [_make_row(rec, None, None, None, qc_flag=f"zip_read_error:{type(exc).__name__}")]


def reduce_to_one_row_per_subject_visit_run(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    found = df[df["scan_date"].notna()].copy()
    if found.empty:
        return found

    match_rank = {"exact_or_strong_match": 0, "contains_restingstate": 1}
    found["match_rank"] = found["match_type"].map(match_rank).fillna(9)

    found = found.sort_values(
        by=["pseudonym", "visit", "run", "match_rank", "scan_date", "source_member"],
        ascending=True,
    )

    reduced = found.groupby(["pseudonym", "visit", "run"], as_index=False).first()
    reduced = reduced.drop(columns=["match_rank"], errors="ignore")
    return reduced


print("Helper functions loaded (optimized).")

Helper functions loaded (optimized).


In [3]:
# ── Build inventory of ZIP archives ──────────────────────────────────────────
inventory: list[ArchiveRecord] = []
for visit_dir in tqdm(list(iter_visit_dirs(ALL_DATA_ROOT)), desc="Scanning visit dirs"):
    for zpath in sorted(visit_dir.glob("*.zip")):
        rec = parse_zip_name(zpath)
        if rec is not None:
            inventory.append(rec)

print(f"Found {len(inventory)} ZIP archives across all visits.\n")

# ── Extract dates (ProcessPoolExecutor, 6 workers — optimal for GPFS I/O) ──
MAX_WORKERS = 6

all_rows: list[dict] = []
errors: list[str] = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(extract_record_from_zip, rec): rec for rec in inventory}

    for future in tqdm(cf.as_completed(futures), total=len(futures),
                       desc="Extracting dates", unit="zip"):
        rec = futures[future]
        try:
            rows = future.result()
            all_rows.extend(rows)
        except Exception as exc:
            errors.append(f"{rec.zip_path.name}: {type(exc).__name__}: {exc}")

if errors:
    print(f"\n⚠  {len(errors)} unexpected errors:")
    for e in errors[:10]:
        print(f"   {e}")

# ── Build DataFrame, reduce, sort ───────────────────────────────────────────
df_raw = pd.DataFrame(all_rows)
df = reduce_to_one_row_per_subject_visit_run(df_raw)

visit_sort_key = df["visit"].map(VISIT_RANK)
df = df.assign(_vsort=visit_sort_key).sort_values(
    by=["pseudonym", "_vsort", "run"]
).drop(columns=["_vsort"]).reset_index(drop=True)

# ── Export to Excel (separate sheets for T1_01 / T1_02) ─────────────────────
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for run_label in RUNS_OF_INTEREST:
        sheet = df[df["run"] == run_label]
        sheet.to_excel(writer, sheet_name=run_label, index=False)

print(f"\n✅ Saved {len(df)} rows → {OUTPUT_XLSX}")
print(f"   Sheets: {RUNS_OF_INTEREST}")
print(f"   Unique pseudonyms: {df['pseudonym'].nunique()}")
print(f"   Visits present: {sorted(df['visit'].unique(), key=lambda v: VISIT_RANK.get(v, 99))}")

Scanning visit dirs:   0%|          | 0/6 [00:00<?, ?it/s]

Scanning visit dirs: 100%|██████████| 6/6 [00:00<00:00, 141.62it/s]

Found 3516 ZIP archives across all visits.




Extracting dates: 100%|██████████| 3516/3516 [01:08<00:00, 51.29zip/s] 



✅ Saved 3391 rows → /dss/dsshome1/0A/di54lup/DELCODE/Delcode-Scan-Dates/restingstate_scan_dates_M0_M60.xlsx
   Sheets: {'T1_02', 'T1_01'}
   Unique pseudonyms: 965
   Visits present: ['M0', 'M12', 'M24', 'M36', 'M48', 'M60']


In [4]:
# Combine T1_01 and T1_02 into one sheet, keeping subject/date order
sheet_a = pd.read_excel(OUTPUT_XLSX, sheet_name="T1_01")
sheet_b = pd.read_excel(OUTPUT_XLSX, sheet_name="T1_02")

combined = pd.concat([sheet_a, sheet_b], ignore_index=True)

# Stable order: subject -> visit order -> scan date -> run
combined["_visit_rank"] = combined["visit"].map(VISIT_RANK).fillna(999)
combined["_scan_date_dt"] = pd.to_datetime(combined["scan_date"], errors="coerce")

combined = combined.sort_values(
    by=["pseudonym", "_visit_rank", "_scan_date_dt", "run"],
    ascending=[True, True, True, True],
).drop(columns=["_visit_rank", "_scan_date_dt"])

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    combined.to_excel(writer, sheet_name="T1_01_T1_02", index=False)

print(f"✅ Combined sheet written: {OUTPUT_XLSX} [T1_01_T1_02] ({len(combined)} rows)")

✅ Combined sheet written: /dss/dsshome1/0A/di54lup/DELCODE/Delcode-Scan-Dates/restingstate_scan_dates_M0_M60.xlsx [T1_01_T1_02] (3391 rows)


In [5]:
# Optional QC summary
if df_raw.empty:
    print("No rows extracted.")
else:
    print("\nQC: raw flags")
    print(df_raw["qc_flag"].fillna("(none)").value_counts(dropna=False).head(20))

if df.empty:
    print("No final scan dates found.")
else:
    print("\nRows per run in final output:")
    print(df["run"].value_counts())

    print("\nRows per visit in final output:")
    print(df["visit"].value_counts().reindex(VISIT_ORDER).fillna(0).astype(int))

    display(df.head(20))


QC: raw flags
qc_flag
                          3392
restingstate_not_found     124
Name: count, dtype: int64

Rows per run in final output:
run
T1_01    3321
T1_02      70
Name: count, dtype: int64

Rows per visit in final output:
visit
M0     930
M12    752
M24    539
M36    470
M48    408
M60    292
Name: count, dtype: int64


,pseudonym,visit,run,scan_date,date_source_tag,match_type,series_description,protocol_name,sequence_name,series_instance_uid,source_zip,source_member,qc_flag
0,011d501d1,M0,T1_01,2015-10-08,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.36.40601.2015100814474374166...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
1,011d501d1,M12,T1_01,2016-11-11,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.36.40601.3000001611211444402...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
2,01490270a,M0,T1_01,2017-03-01,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.43.66038.3000001703011411379...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
3,01490270a,M12,T1_01,2018-02-07,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.43.66038.2018020715051650417...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
4,01643c48b,M0,T1_01,2014-06-13,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.19.45385.2014061314133117916...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
5,01643c48b,M12,T1_02,2015-06-12,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.19.45385.2015061213111085432...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
6,0165a12e0,M0,T1_01,2014-08-28,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.32.35105.2014082813082144297...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/5-dzne_RestingState_3_5iso/resources/DIC...,
7,0165a12e0,M12,T1_01,2015-09-24,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.43.66038.2015092409352736806...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
8,0165e2425,M12,T1_01,2019-09-28,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.19.45392.2019092816225415602...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/10-dzne_RestingState_3_5iso/resources/DI...,
9,01c968c29,M0,T1_01,2016-06-01,AcquisitionDate,contains_restingstate,dzne_RestingState_3.5iso,dzne_RestingState_3.5iso,*epfid2d1_64,1.3.12.2.1107.5.2.43.66038.2016060114401527064...,/dss/dssfs03/pn72zi/pn72zi-dss-0001/di38jor/Pr...,SCANS/4-dzne_RestingState_3_5iso/resources/DIC...,
